In [ ]:
from skimage.io import imread, imsave
from skimage.segmentation import clear_border
import os
import numpy as np

# Define the directory path containing the images
image_directory = r'D:\Chia_Ling_Yeast_Live_imaging\20240718-ero1 KA ppm1_Tm_EGFP_analysis'
label_directory = r'D:\Chia_Ling_Yeast_Live_imaging\20240718-ero1 KA ppm1_Tm_EGFP_analysis'
output_directory = r'D:\Chia_Ling_Yeast_Live_imaging\20240718-ero1 KA ppm1_Tm_EGFP_analysis'

# Define the specific naming patterns for images and labels
image_pattern = "_intensity.tif"
label_pattern = "_label.tif"

# List all image files in the directory that match the specified naming pattern
image_files = [f for f in os.listdir(image_directory) if f.endswith(image_pattern) and os.path.isfile(os.path.join(image_directory, f))]
label_files = [f for f in os.listdir(label_directory) if f.endswith(label_pattern) and os.path.isfile(os.path.join(label_directory, f))]

# Sort files to ensure matching order of images and labels if necessary
image_files.sort()
label_files.sort()

# Iterate over the files and load them
for img_file, label_file in zip(image_files, label_files):
  img_path = os.path.join(image_directory, img_file)
  label_path = os.path.join(label_directory, label_file)
  
  img_time_series = imread(img_path)
  img_label_time_series = imread(label_path)

  base_filename, file_extension = os.path.splitext(img_file)
  
  # Construct the output filename
  base_filename = base_filename.replace('_intensity', '')
  output_filename = f"{base_filename}_filtered_label{file_extension}"

  # Define the full output path
  output_path = os.path.join(output_directory, output_filename)
      
  unique_labels_per_time_point = []

  # New array to store labels with edge objects removed
  edge_removed_labels = np.zeros_like(img_label_time_series)

  for t in range(img_label_time_series.shape[0]): # 'Z', Y, Z
      label_image = img_label_time_series[t, :, :]
      
      # Remove labels on the edge using clear_border directly
      label_image_no_edge = clear_border(label_image)
      
      # Store the edge-removed label image
      edge_removed_labels[t, :, :] = label_image_no_edge
      
      unique_labels = np.unique(label_image_no_edge) # Extract the unique label IDs
      # Remove 0 (background) from the unique labels
      unique_labels = set(unique_labels) - {0}
      unique_labels_per_time_point.append(unique_labels)

  # Start with the set of labels from the first time point
  consistent_labels = set(unique_labels_per_time_point[0])

  # Iterate through the list of unique labels per time point and keep only the intersection
  for labels in unique_labels_per_time_point[1:]:
      consistent_labels = consistent_labels.intersection(labels)

  # Initialize an empty array to store the filtered images
  filtered_time_series_label_images = np.zeros_like(img_label_time_series)

  # Loop through each time point in the time series
  for t in range(img_label_time_series.shape[0]):
      # Extract the label image for the current time point (now using edge_removed_labels)
      label_image = edge_removed_labels[t, :, :]
  
      # Create a mask that is True for pixels with labels that are in 'consistent_labels'
      mask = np.isin(label_image, list(consistent_labels))
  
      # Apply the mask, retaining only the pixels with consistent labels
      # Pixels not in 'consistent_labels' are set to 0 (or another background value as needed)
      filtered_label_image = np.where(mask, label_image, 0)
  
      # Store the filtered label image in the corresponding slice of the new 3D array
      filtered_time_series_label_images[t, :, :] = filtered_label_image

  # Save the new filtered time-series label images to a multi-dimensional TIFF file
  imsave(output_path, filtered_time_series_label_images)

# Created/Modified files during execution:
print(output_path)